In [1]:
import geopandas as gpd
import numpy as np
import rasterio
from rasterstats import zonal_stats
import glob
import pandas as pd
import os
from datetime import datetime
import re
import matplotlib.pyplot as plt

In [2]:
trees_path = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Code\\thesis\\Output\\schilderswijk_2.5_1.5_3_polygons.gpkg"
grid_path = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\den_haag_grid_extent.gpkg"


trees = gpd.read_file(trees_path)
grid = gpd.read_file(grid_path)

### Grid voor de tree properties

In [3]:
if trees.crs != grid.crs:
    trees = trees.to_crs(grid.crs)

# Voeg een uniek ID toe aan het grid als dat er nog niet is
if "grid_id" not in grid.columns:
    grid["grid_id"] = grid.index

# Sla het originele oppervlak van de boomkroon-polygonen op
trees["orig_area"] = trees.geometry.area

# Snijd de boomkronen op per gridcel (Intersection)
inter = gpd.overlay(trees, grid, how="intersection")

# Herbereken het volume naar verhouding van het geknipte oppervlak
inter["clipped_area"] = inter.geometry.area
inter["area_ratio"] = inter["clipped_area"] / inter["orig_area"]

# Volume, oppervlakte en biomassa
inter["boomkroonvolume m3"] = (inter["measured_cv_m3"] * inter["area_ratio"]).round(2)
inter["boomkroonoppervlakte m2"] = (inter['measured_ca_m2'] * inter['area_ratio']).round(2)
inter["biomassa_bermudez kg"] = (inter['model_agb_kg_bermudez'] * inter['area_ratio']).round(2)

# Sommeer per gridcell
boomkroonvolume_m3 = inter.groupby("grid_id")["boomkroonvolume m3"].sum().reset_index()
boomkroonoppervlakte_m2 = inter.groupby("grid_id")["boomkroonoppervlakte m2"].sum().reset_index()
biomassa_kg = inter.groupby("grid_id")["biomassa_bermudez kg"].sum().reset_index()


# Koppel het berekende volume terug aan het oorspronkelijke grid (inclusief temperatuur)
final_grid = grid.merge(boomkroonvolume_m3, on="grid_id", how="left")
final_grid = final_grid.merge(boomkroonoppervlakte_m2, on="grid_id", how="left")
final_grid = final_grid.merge(biomassa_kg, on="grid_id", how="left")

# Vul cellen zonder bomen op met 0 m³ volume
final_grid["boomkroonvolume m3"] = final_grid["boomkroonvolume m3"].fillna(0)
final_grid["boomkroonoppervlakte m2"] = final_grid["boomkroonoppervlakte m2"].fillna(0)
final_grid["biomassa_bermudez kg"] = final_grid["biomassa_bermudez kg"].fillna(0)

# Sla op als nieuw bestand
output_path = trees_path.replace(".gpkg", "_ecostress.gpkg")
final_grid.to_file(output_path, driver="GPKG")

print("Succesvol gecombineerd! Resultaat opgeslagen in:", output_path)

C:\Users\hoeks\AppData\Local\Temp\ipykernel_14516\415921812.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  trees["orig_area"] = trees.geometry.area
C:\Users\hoeks\AppData\Local\Temp\ipykernel_14516\415921812.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  inter["clipped_area"] = inter.geometry.area


Succesvol gecombineerd! Resultaat opgeslagen in: C:\Floor\Documenten\ENV-year-2\Thesis\Code\thesis\Output\schilderswijk_2.5_1.5_3_polygons_ecostress.gpkg


### Combine all the ECOSTRESS data in one excel and geopackage

In [30]:
folder_ecostress = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\ECOSTRESS\\ECOSTRESS_sharp\\sharp\\"
output_csv = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\ecostress_merged.csv"
output_gpkg = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\ecostress_merged.gpkg"

# 2. Laad basis-grid
print("Basis-grid laden...")
grid = gpd.read_file(grid_path)
print(f"-> Grid geladen met {len(grid)} cellen.")

# 3. TIF-bestanden zoeken
tif_files = sorted(glob.glob(os.path.join(folder_ecostress, "*.tif")))
if len(tif_files) == 0:
    # Probeer ook hoofdletters indien nodig
    tif_files = sorted(glob.glob(os.path.join(folder_ecostress, "*.TIF")))

print(f"-> Aantal gevonden TIF-bestanden: {len(tif_files)}")

if len(tif_files) == 0:
    print(
        "FOUT: Geen .tif bestanden gevonden! Controleer het pad 'folder_ecostress'."
    )
else:
    # 4. Zorg dat het grid exact hetzelfde coördinatensysteem (CRS) heeft als de rasters
    with rasterio.open(tif_files[0]) as src:
        raster_crs = src.crs

    if grid.crs != raster_crs:
        print(
            f"-> CRS aanpassen van grid ({grid.crs}) naar raster CRS ({raster_crs})..."
        )
        grid = grid.to_crs(raster_crs)

    # 5. Maak een lijst van (x, y) tuples voor de centroids van alle gridcellen
    centroids = [(geom.centroid.x, geom.centroid.y) for geom in grid.geometry]

    # 6. Lus door alle TIFs en sample de temperaturen
    for i, tif_path in enumerate(tif_files, 1):
        filename = os.path.basename(tif_path)

        # Bepaal kolomnaam op basis van datum/tijd (bijv. LST_20250602T135437)
        match = re.search(r"\d{8}T\d{6}", filename)
        if match:
            dt = datetime.strptime(match.group(0), "%Y%m%dT%H%M%S")
            col_name = f"LST_{dt.strftime('%Y-%m-%d %H:%M')}"
        else:
            col_name = f"LST_{i}"

        print(f"[{i}/{len(tif_files)}] Verwerken: {col_name}...")

        with rasterio.open(tif_path) as src:
            # Sample de pixelwaarden op de centroid-locaties
            sampled_values = [val[0] for val in src.sample(centroids)]
            nodata_val = src.nodata

            celsius_list = []
            for val in sampled_values:
                # Sla NoData / ongeldige waarden over
                if (
                    val is None
                    or (nodata_val is not None and val == nodata_val)
                    or np.isnan(val)
                ):
                    celsius_list.append(None)
                else:
                    # Omrekenen van Kelvin naar Celsius
                    celsius_list.append(round(val - 273.15, 2))

            # Voeg toe als nieuwe kolom aan het dataframe
            grid[col_name] = celsius_list

    # 7. Exporteer resultaten
    print("Bezig met opslaan...")

    # Sla op als CSV (zonder geometriekolom voor makkelijke verwerking in Excel/SPSS/R)
    df_export = grid.drop(columns=["geometry", "Temp"])
    df_export.to_csv(output_csv, index=False)

    # Sla op als GeoPackage (met ruimtelijke polygonen voor QGIS)
    grid = grid.drop(columns=["Temp"])  # Verwijder de oude 'Temp' kolom als die er is
    grid.to_file(output_gpkg, driver="GPKG")

    print(f"Klaar! CSV opgeslagen in: {output_csv}")
    print(f"GeoPackage opgeslagen in: {output_gpkg}")

Basis-grid laden...
-> Grid geladen met 249483 cellen.
-> Aantal gevonden TIF-bestanden: 31
[1/31] Verwerken: LST_2025-06-02 13:54...
[2/31] Verwerken: LST_2025-06-02 15:30...
[3/31] Verwerken: LST_2025-06-03 13:05...
[4/31] Verwerken: LST_2025-06-04 12:18...
[5/31] Verwerken: LST_2025-06-05 13:06...
[6/31] Verwerken: LST_2025-06-06 12:17...
[7/31] Verwerken: LST_2025-06-09 11:29...
[8/31] Verwerken: LST_2025-06-11 11:29...
[9/31] Verwerken: LST_2025-06-12 09:04...
[10/31] Verwerken: LST_2025-06-15 09:51...
[11/31] Verwerken: LST_2025-06-16 09:03...
[12/31] Verwerken: LST_2025-06-18 07:26...
[13/31] Verwerken: LST_2025-06-29 04:58...
[14/31] Verwerken: LST_2025-06-30 04:10...
[15/31] Verwerken: LST_2025-07-10 22:26...
[16/31] Verwerken: LST_2025-07-11 21:37...
[17/31] Verwerken: LST_2025-07-25 16:45...
[18/31] Verwerken: LST_2025-07-30 15:55...
[19/31] Verwerken: LST_2025-08-06 11:52...
[20/31] Verwerken: LST_2025-08-09 12:39...
[21/31] Verwerken: LST_2025-08-10 11:50...
[22/31] Verwer

### Plot the ECOSTRESS data per gridcell

In [4]:
# output_csv = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\ecostress_merged.csv"

# df = pd.read_csv(output_csv)

# print(df.columns.tolist())

# lst_columns = [col for col in df.columns if col.startswith("LST_")]

output_csv = "C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\ecostress_merged.csv"

df = pd.read_csv(output_csv, sep=";")

print(df.columns.tolist())

lst_columns = [col for col in df.columns if col.startswith("LST_")]

times = [
    datetime.strptime(col.replace("LST_", ""), "%Y-%m-%d %H:%M")
    for col in lst_columns
]

plt.figimage(figsize=(12, 6))
plt.plot(times, df[lst_columns].mean(), marker="o")



['LST_2025-06-02 13:54', 'LST_2025-06-02 15:30', 'LST_2025-06-03 13:05', 'LST_2025-06-04 12:18', 'LST_2025-06-05 13:06', 'LST_2025-06-06 12:17', 'LST_2025-06-09 11:29', 'LST_2025-06-11 11:29', 'LST_2025-06-12 09:04', 'LST_2025-06-15 09:51', 'LST_2025-06-16 09:03', 'LST_2025-06-18 07:26', 'LST_2025-06-29 04:58', 'LST_2025-06-30 04:10', 'LST_2025-07-10 22:26', 'LST_2025-07-11 21:37', 'LST_2025-07-25 16:45', 'LST_2025-07-30 15:55', 'LST_2025-08-06 11:52', 'LST_2025-08-09 12:39', 'LST_2025-08-10 11:50', 'LST_2025-08-11 09:25', 'LST_2025-08-13 09:24', 'LST_2025-08-13 11:01', 'LST_2025-08-14 10:12', 'LST_2025-08-15 07:47', 'LST_2025-08-19 06:10', 'LST_2025-08-21 06:10', 'LST_2025-08-21 07:47', 'LST_2025-08-23 06:10', 'LST_2025-08-26 03:44']


TypeError: figimage() missing 1 required positional argument: 'X'

In [ ]:
import matplotlib.animation as animation

# Laad je GeoPackage
grid = gpd.read_file("C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\ecostress_merged.gpkg")

# Selecteer LST-kolommen
lst_cols = [c for c in grid.columns if c.startswith("LST_")]

# Bepaal tijden voor de x-as van de titel
times = [c[4:] for c in lst_cols]  # "2025-06-02 13:54" etc.

# Plot-setup
fig, ax = plt.subplots(figsize=(10, 8))
grid.plot(ax=ax, color="white", edgecolor="none")  # witte achtergrond

all_vals = grid[lst_cols].values.flatten()
vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)

# Functie per frame
def animate(i):
    ax.clear()
    grid.plot(column=lst_cols[i], ax=ax, cmap="YlOrRd", edgecolor="none", legend=True)
    ax.set_title(times[i], fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    return [ax]

# Animatie maken
anim = animation.FuncAnimation(fig, animate, frames=len(lst_cols), interval=500)

# Opslaan als mp4 (vereist ffmpeg)
anim.save("C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\lst_animation.mp4", writer="ffmpeg", fps=2)

# Of als GIF (geen ffmpeg nodig)
# anim.save("C:\\Floor\\Documenten\\ENV-year-2\\Thesis\\Data\\lst_animation.gif", writer="pillow", fps=2)

plt.show()

KeyboardInterrupt: 